## Section 1: Imports
Load all required libraries — TensorFlow, Transformers (DistilBERT), sklearn, NLTK, plotting, etc.

In [ ]:
# Import all required libraries for data processing, deep learning, and visualization
import os, re, ssl, warnings, json
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')          # non-interactive backend so plots save to files
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
ssl._create_default_https_context = ssl._create_unverified_context   # fix SSL for NLTK downloads
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem   import WordNetLemmatizer

import tensorflow as tf
from transformers import (
    DistilBertTokenizer,
    TFDistilBertForSequenceClassification,
    create_optimizer,
)

from sklearn.model_selection   import train_test_split
from sklearn.metrics           import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from collections import Counter

warnings.filterwarnings('ignore')
print("All imports successful ✅")
print(f"TensorFlow Version: {tf.__version__}")

# Check for GPU (Metal acceleration)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
        print("GPU (Metal) acceleration is ACTIVE ✅")
    except RuntimeError as e:
        print(e)
else:
    print("GPU is NOT available, using CPU. ⚠️")

## Section 2: Configuration
Set file paths, model hyper-parameters, label mapping, and environment configuration.

In [ ]:
# --- Paths ---
BASE_DIR   = os.path.dirname(os.path.abspath(''))   # project root
DATA_DIR   = os.path.join(BASE_DIR, 'Ecommerce_dataset')
OUTPUT_DIR = os.path.join(BASE_DIR, 'personal_update', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH       = os.path.join(DATA_DIR, 'train_data.csv')
TEST_PATH        = os.path.join(DATA_DIR, 'test_data.csv')
TEST_HIDDEN_PATH = os.path.join(DATA_DIR, 'test_data_hidden.csv')

# --- Model hyper-parameters ---
MODEL_NAME       = 'distilbert-base-uncased'   # lighter, faster BERT variant
MAX_LENGTH       = 128                          # max tokens per review
BATCH_SIZE       = 16
EPOCHS           = 4
LEARNING_RATE    = 2e-5
VALIDATION_SPLIT = 0.15                         # 15 % held out for validation
RANDOM_SEED      = 42

# --- Label mapping ---
LABEL_MAP   = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']

print(f"Configuration loaded for {MODEL_NAME}")

## Section 3: Data Loading
Read the three CSV files (train, test, test_hidden) and inspect shapes, sentiment distribution, and missing values.

In [ ]:
# Load the three datasets
train_df       = pd.read_csv(TRAIN_PATH)
test_df        = pd.read_csv(TEST_PATH)
test_hidden_df = pd.read_csv(TEST_HIDDEN_PATH)

print(f"Training data:    {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test data:        {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"Test hidden data: {test_hidden_df.shape[0]} rows, {test_hidden_df.shape[1]} columns")

print(f"\nTraining sentiment distribution:")
print(train_df['sentiment'].value_counts())
print(f"\nMissing values in training data:")
print(train_df.isnull().sum())

## Section 4: Text Preprocessing
Light cleaning for BERT (lowercase, remove URLs/HTML/special chars). Also a heavier version for ABSA that removes stopwords and lemmatizes.

In [ ]:
# Build stopword set but KEEP negation words (critical for sentiment)
stop_words = set(stopwords.words('english'))
negation_words = {'not','no','nor','neither','never','none',
                  "don't","doesn't","didn't","won't","wouldn't",
                  "can't","cannot","couldn't","shouldn't","isn't",
                  "aren't","wasn't","weren't","hasn't","haven't"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Light cleaning for BERT — keep structure, just remove noise."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # URLs
    text = re.sub(r'<.*?>', '', text)               # HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)      # non-letters
    text = re.sub(r'\s+', ' ', text).strip()       # extra whitespace
    return text

def preprocess_for_analysis(text):
    """Heavier preprocessing for ABSA / word clouds (stopword removal + lemmatization)."""
    text = clean_text(text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Apply light cleaning and combine title + text for richer BERT input
for df in [train_df, test_df, test_hidden_df]:
    df['clean_text']    = df['reviews.text'].apply(clean_text)
    df['reviews.title'] = df['reviews.title'].fillna('')
    df['combined_text'] = (df['reviews.title'].apply(clean_text) + ' ' + df['clean_text']).str.strip()

# Encode sentiment labels as integers
train_df['label']       = train_df['sentiment'].map(LABEL_MAP)
test_hidden_df['label'] = test_hidden_df['sentiment'].map(LABEL_MAP)

print(f"Sample cleaned text:")
print(f"  Original:  {train_df['reviews.text'].iloc[0][:100]}...")
print(f"  Cleaned:   {train_df['combined_text'].iloc[0][:100]}...")
print(f"\nPreprocessing complete for {len(train_df)} training samples.")

## Section 5: Train / Validation Split
Stratified split so each sentiment class keeps the same ratio in both sets.

In [ ]:
# Stratified split — keeps class proportions identical in train & val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['combined_text'].values,
    train_df['label'].values,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    stratify=train_df['label'].values,
)

print(f"Training samples:   {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"\nTraining label distribution:")
for name, idx in LABEL_MAP.items():
    count = (train_labels == idx).sum()
    print(f"  {name}: {count} ({count/len(train_labels)*100:.1f}%)")

## Section 6: Compute Class Weights
Handle class imbalance — minority classes (Negative) get higher weight so the model doesn't ignore them.

In [ ]:
# Compute balanced class weights so minority classes get more attention
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_labels,
)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}

print("Class weights (higher = model pays more attention):")
for name, weight in zip(LABEL_NAMES, class_weights):
    print(f"  {name}: {weight:.4f}")
print(f"\nNegative reviews get ~{class_weights[0]/class_weights[2]:.1f}x more weight than Positive")

## Section 7: Tokenization & TensorFlow Datasets
Convert review text into token IDs and wrap in tf.data.Dataset for efficient training.

In [ ]:
# Load the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

def get_tf_dataset(texts, labels=None, shuffle=False):
    """Convert texts and optional labels to a batched tf.data.Dataset."""
    encodings = tokenizer(
        list(texts),
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='tf',
    )
    
    # Convert to dict for TF model inputs
    dataset_dict = {
        'input_ids':      encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
    }
    
    if labels is not None:
        dataset = tf.data.Dataset.from_tensor_slices((dataset_dict, labels))
    else:
        dataset = tf.data.Dataset.from_tensor_slices(dataset_dict)
        
    if shuffle:
        dataset = dataset.shuffle(len(texts))
    
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Build datasets
train_dataset = get_tf_dataset(train_texts, train_labels, shuffle=True)
val_dataset   = get_tf_dataset(val_texts,   val_labels)

# Quick sanity check
for batch_features, batch_labels in train_dataset.take(1):
    print(f"Sample batch input_ids shape: {batch_features['input_ids'].shape}")
    print(f"Sample batch labels shape:    {batch_labels.shape}")
    break

print(f"\nDatasets created: {len(train_texts)} train, {len(val_texts)} validation")

## Section 8: Build & Train the Model
Load pre-trained TF DistilBERT, add a 3-class classification head, and fine-tune using Keras API.

In [ ]:
# Load TF DistilBERT for Sequence Classification
model = TFDistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Optimizer with weight decay and linear warmup
num_train_steps = len(train_dataset) * EPOCHS
optimizer, lr_schedule = create_optimizer(
    init_lr=LEARNING_RATE,
    num_train_steps=num_train_steps,
    num_warmup_steps=int(0.1 * num_train_steps),
    weight_decay_rate=0.01,
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Custom callback to track F1-score on validation set after each epoch
class F1MetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_data, val_labels):
        super().__init__()
        self.val_data   = val_data
        self.val_labels = val_labels
        self.history    = {'val_f1': []}

    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation data
        preds_logits = self.model.predict(self.val_data, verbose=0).logits
        preds = np.argmax(preds_logits, axis=1)
        
        f1 = f1_score(self.val_labels, preds, average='weighted')
        self.history['val_f1'].append(f1)
        logs['val_f1'] = f1
        print(f" - val_f1: {f1:.4f}")

f1_callback = F1MetricsCallback(val_dataset, val_labels)

print(f"Model: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"Total training steps: {num_train_steps}\n")

# Train with class weights
res = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights_dict,
    callbacks=[f1_callback]
)

# Consolidate history
history = {
    'train_loss':   res.history['loss'],
    'val_loss':     res.history['val_loss'],
    'val_accuracy': res.history['val_accuracy'],
    'val_f1':       f1_callback.history['val_f1']
}

## Section 9: Evaluation on Validation Set
Print a full classification report and save the confusion matrix as a PNG.

In [ ]:
# Get final validation predictions
val_preds_logits = model.predict(val_dataset).logits
val_preds = np.argmax(val_preds_logits, axis=1)

print("Classification Report (Validation Set):")
print(classification_report(val_labels, val_preds, target_names=LABEL_NAMES))

val_f1_final = f1_score(val_labels, val_preds, average='weighted')
print(f"*** Weighted F1-Score: {val_f1_final:.4f} ***")
if val_f1_final > 0.85:
    print("✅ TARGET MET: F1-Score > 85%")
else:
    print("⚠️  F1-Score below 85% target — consider more epochs or tuning")

# Confusion matrix heatmap
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix — Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_confusion_matrix_validation.png'), dpi=150)
plt.show()
print("Saved to outputs/01_confusion_matrix_validation.png")

## Section 10: Predict on Test Set & Evaluate
Run the trained model on the hidden test set and save predictions.

In [ ]:
# Evaluate on the hidden test set
test_hidden_labels = test_hidden_df['label'].values
test_hidden_dataset = get_tf_dataset(test_hidden_df['combined_text'].values, test_hidden_labels)

test_preds_logits = model.predict(test_hidden_dataset).logits
test_preds = np.argmax(test_preds_logits, axis=1)

print("Classification Report (Test Set):")
print(classification_report(test_hidden_labels, test_preds, target_names=LABEL_NAMES))

test_f1  = f1_score(test_hidden_labels, test_preds, average='weighted')
test_acc = accuracy_score(test_hidden_labels, test_preds)
print(f"*** Test Weighted F1-Score: {test_f1:.4f} ***")
print(f"*** Test Accuracy:          {test_acc:.4f} ***")

# Confusion matrix for test set
cm_test = confusion_matrix(test_hidden_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix — Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_confusion_matrix_test.png'), dpi=150)
plt.show()
print("Saved to outputs/02_confusion_matrix_test.png")

# Also predict on the unlabelled test_data.csv and save to CSV
test_nolabel_dataset = get_tf_dataset(test_df['combined_text'].values)
final_preds_logits = model.predict(test_nolabel_dataset).logits
final_preds = np.argmax(final_preds_logits, axis=1)

test_df_output = test_df.copy()
test_df_output['predicted_sentiment'] = [LABEL_NAMES[p] for p in final_preds]
test_df_output[['name','reviews.text','reviews.title','predicted_sentiment']].to_csv(
    os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False
)
print("Predictions saved to outputs/test_predictions.csv")

## Section 11: Training History Visualization
Plot loss, F1-score, and accuracy across epochs.

In [ ]:
# Visualize evolution over epochs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, EPOCHS+1), history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(range(1, EPOCHS+1), history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Training & Validation Loss');  axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('Loss')
axes[0].legend();  axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS+1), history['val_f1'], 'g-o', label='Val F1-Score')
axes[1].axhline(y=0.85, color='r', linestyle='--', label='Target (0.85)')
axes[1].set_title('Validation F1-Score per Epoch');  axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('F1-Score')
axes[1].legend();  axes[1].grid(True, alpha=0.3)

axes[2].plot(range(1, EPOCHS+1), history['val_accuracy'], 'm-o', label='Val Accuracy')
axes[2].set_title('Validation Accuracy per Epoch');  axes[2].set_xlabel('Epoch');  axes[2].set_ylabel('Accuracy')
axes[2].legend();  axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_training_history.png'), dpi=150)
plt.show()
print("Saved to outputs/03_training_history.png")

## Section 12: Aspect-Based Sentiment Analysis (ABSA)
Identify product aspects and sentiment breakdown.

In [ ]:
ASPECT_KEYWORDS = {
    'Price/Value':       ['price','cost','expensive','cheap','value','money','worth','deal','affordable','budget','sale','bargain'],
    'Screen/Display':    ['screen','display','resolution','bright','hd','visual','color','picture','pixel','view'],
    'Battery/Power':     ['battery','charge','charging','power','last','outlet','plug','cord','usb'],
    'Sound/Audio':       ['sound','speaker','audio','volume','music','loud','bass','hear','listen','noise'],
    'Speed/Performance': ['speed','fast','slow','lag','performance','quick','responsive','processor','ram','memory'],
    'Build/Design':      ['build','quality','durable','sturdy','design','weight','light','heavy','size','compact','portable'],
    'Ease of Use':       ['easy','simple','intuitive','user-friendly','setup','navigate','interface','learn','beginner','convenient'],
    'Apps/Software':     ['app','apps','software','store','download','install','update','google','play','alexa','skill'],
    'Camera':            ['camera','photo','picture','video','record','selfie'],
    'Kids/Family':       ['kid','kids','child','children','son','daughter','grandkid','family','parent','parental','toddler'],
}

def extract_aspects(text):
    text_lower = text.lower()
    return [asp for asp, kws in ASPECT_KEYWORDS.items() if any(kw in text_lower for kw in kws)]

train_df['aspects'] = train_df['reviews.text'].apply(extract_aspects)
all_aspects   = [a for aspects in train_df['aspects'] for a in aspects]
aspect_counts = Counter(all_aspects)

aspect_sentiment_data = []
for aspect in ASPECT_KEYWORDS:
    mask   = train_df['aspects'].apply(lambda x: aspect in x)
    subset = train_df[mask]
    if len(subset) > 0:
        dist = subset['sentiment'].value_counts(normalize=True) * 100
        pos, neu, neg = dist.get('Positive',0), dist.get('Neutral',0), dist.get('Negative',0)
        aspect_sentiment_data.append({'Aspect': aspect, 'Positive': pos, 'Neutral': neu, 'Negative': neg, 'Total Reviews': len(subset)})

aspect_df = pd.DataFrame(aspect_sentiment_data)

plt.figure(figsize=(12, 6))
heatmap_data = aspect_df.set_index('Aspect')[['Positive','Neutral','Negative']]
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', center=50, linewidths=0.5)
plt.title('Aspect-Based Sentiment Analysis — Sentiment % by Aspect')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_absa_heatmap.png'), dpi=150)
plt.show()

## Section 13: Category-Based Product Comparison
Compare sentiment across product categories.

In [ ]:
def categorize_product(name):
    n = name.lower()
    if 'echo show' in n:   return 'Echo Show'
    if 'echo plus' in n:   return 'Echo Plus'
    if 'tap' in n:         return 'Amazon Tap'
    if 'fire kids' in n:   return 'Fire Kids Tablet'
    if 'fire hd 10' in n:  return 'Fire HD 10'
    if 'fire hd 8' in n or 'fire hd8' in n: return 'Fire HD 8'
    if 'fire' in n and 'tablet' in n:        return 'Fire 7 Tablet'
    if 'oasis' in n:       return 'Kindle Oasis'
    if 'voyage' in n:      return 'Kindle Voyage'
    if 'kindle' in n:      return 'Kindle E-reader'
    if 'fire tv' in n:     return 'Fire TV'
    return 'Other'

train_df['product_category'] = train_df['name'].apply(categorize_product)
cat_sentiment = train_df.groupby('product_category')['sentiment'].value_counts(normalize=True).unstack(fill_value=0) * 100

plt.figure(figsize=(10, 6))
cat_sentiment[['Positive','Neutral','Negative']].plot(kind='barh', stacked=True, color=['#2ecc71','#f39c12','#e74c3c'])
plt.title('Sentiment Distribution by Product Category')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_category_comparison.png'), dpi=150)
plt.show()

## Section 14: Overall Sentiment Visualizations
Pie chart and review length distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

sentiment_counts = train_df['sentiment'].value_counts()
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Overall Sentiment Distribution')

train_df['text_length'] = train_df['reviews.text'].str.len()
for sent, color in zip(['Positive','Neutral','Negative'], colors):
    subset = train_df[train_df['sentiment'] == sent]['text_length']
    axes[1].hist(subset, bins=50, alpha=0.6, label=sent, color=color)
axes[1].set_title('Review Length by Sentiment')
axes[1].set_xlabel('Length'); axes[1].set_ylabel('Count'); axes[1].legend()
axes[1].set_xlim(0, 1000)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_sentiment_overview.png'), dpi=150)
plt.show()

## Section 15: Word Clouds
Generate word clouds for each sentiment class.

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for idx, (sent, cmap) in enumerate([('Positive','Greens'), ('Neutral','Oranges'), ('Negative','Reds')]):
        text = ' '.join(train_df[train_df['sentiment'] == sent]['reviews.text'].apply(preprocess_for_analysis))
        wc = WordCloud(width=600, height=300, background_color='white', colormap=cmap, max_words=80).generate(text)
        axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].set_title(f'{sent} Reviews')
        axes[idx].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '08_word_clouds.png'), dpi=150)
    plt.show()
except Exception as e:
    print(f"Word cloud generation skipped: {e}")

## Section 16: Save Results Summary
Export all key metrics and training history to JSON.

In [ ]:
results_summary = {
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'validation_f1_score':  round(float(val_f1_final), 4),
    'test_f1_score':        round(float(test_f1), 4),
    'test_accuracy':        round(float(test_acc), 4),
    'training_history':     {k: [float(v) for v in vals] for k, vals in history.items()},
}

with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results summary saved to outputs/results_summary.json")
print(f"🎉 PIPELINE COMPLETE | Final Val F1: {val_f1_final:.4f}")